In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ydata_profiling import ProfileReport
import pickle

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 50

data_folder = "../data/"
images_folder = "../data/images/"
pipelines_folder = "../pipelines/"
df_total = pd.read_csv(os.path.join(data_folder, 'items_phase_1.csv'))
df_train = pd.read_csv(os.path.join(data_folder, 'items_train.csv'))
df_task_1 = pd.read_csv(os.path.join(data_folder, 'task_1.csv'))

# Notebook `items_phase_1.csv`

In [2]:
# profile = ProfileReport(df_total, title="items_phase_1", explorative=True)
# profile.to_notebook_iframe()

In [3]:
df_total.sample(4)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
115694,162697,58.95,780,['1'],NaN,Scarpe basse Rieker,Rieker Scarpe basse N4316-90 Argento m=1P,it
28737,1065401,5130.00,777,['42'],NaN,Zimní bunda Karl Lagerfeld Kids,Karl Lagerfeld Kids Zimní bunda Z30234 D Zlatá...,cz
116872,87199,9.90,"771,772",['42'],NaN,Kapa Jack&Jones Junior,Jack&Jones Junior Kapa 12160311 Tamnoplava,hr
65155,291759,8299.00,230,['1'],NaN,"Sluneční brýle Palm Angels černá barva, PERI09...",- UV filtr 400 poskytuje maximální ochranu pře...,cz


## Key takeaway
- missingy se musi resit u:
    - description (3%)
    - brandEditionTagId (99.8%) - to je asi target?
    - colorTagIdsString (3.1%)
---
# Notebook `task_1.csv`
- kazdy radek je jedna skupina se sloupci item - item4(to jsou id do items_phase_1.csv - itemId)
- Kazdy 

In [4]:
# profile = ProfileReport(df_task_1, title="task_1", explorative=True)
# profile.to_notebook_iframe()

In [5]:
df_task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


---
# Spojeni datasetu


In [6]:
df_task_1.sample()

,item1,item2,item3,item4,item5
6736,52644,111517,293355,456526,729369


In [7]:
df_total.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo'],
      dtype='object')

In [8]:
df_total.geo.unique()

array(['gr', 'lv', 'hr', 'sk', 'hu', 'lt', 'bg', 'it', 'cz', 'si', 'ro',
       'ee', 'pl'], dtype=object)

---
# Dataset `items_train.csv`

In [9]:
print("Total records:", len(df_train))
print("Total null values:\n", df_train.isnull().sum())

Total records: 928234
Total null values:
 itemId                    0
price                     0
colorTagIdsString     27834
departmentIds             0
brandEditionTagId    925518
title                     0
description           35473
geo                       0
label                     0
dtype: int64


In [10]:
# profile = ProfileReport(df_train, title="task_1", explorative=True)
# profile.to_notebook_iframe()


---
# Vycisteni dat 
- prevod na spolecnou menu 
- normalizace meny v danem geo uzemi
- doplneni null hodnot
- null ve sloupci `colorTagIdString` muzu nahradit 0 
- `colorTagIdString` a `departmentIds` je potreba roztrhnout - obsahuji vice hodnot oddelenych carkou

## Tvorba preprocessing pipeline

In [11]:
df_total[df_total["brandEditionTagId"] == 0]

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo


In [12]:
df_train.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo', 'label'],
      dtype='object')

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from PriceGeoTransformer import PriceGeoTransformer
from DepartmentIdsTransformer import DepartmentIdsCleaner
import numpy as np

imput_zero_cols = ['colorTagIdsString', "brandEditionTagId"]
input_zero_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])

input_unknown_cols = ["description"]
input_unknown_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value="Unknown")),
])

categorical_features = ['geo',"price"]
categorical_transformer = Pipeline(steps=[
    ('PriceGeoTransformer', PriceGeoTransformer())
])


# prevadi na stejny format jako jsou barvy - cisla oddelena carkou 
department_features = ['departmentIds']
department_transformer = Pipeline(steps=[
    ('DepartmentIdsCleaner', DepartmentIdsCleaner())
])

# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('zero', input_zero_transformer, imput_zero_cols),
        ('unknown', input_unknown_transformer, input_unknown_cols),
        ('geo', categorical_transformer, categorical_features),
        ('department', department_transformer, department_features)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")



pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

pipeline.fit_transform(df_train)

,colorTagIdsString,brandEditionTagId,description,price_eur,price_scaled,geo,departmentIds,itemId,title,label
0,"230,232",0,Unknown,148.000000,-0.254314,bg,11,692210,Раница Rains 14480 Черен,41599
1,"230,232",0,Rains Раница 14480 Черен,148.000000,-0.254314,bg,11,943360,Раница Rains,41599
2,230,0,Раница от колекция Rains. Моделът е направен о...,179.900000,-0.102645,bg,1,447879,Раница Rains Trail Cord Rolltop Backpack W3 в ...,41599
3,230,0,- Непромокаем модел.\n- Повишена водоустойчиво...,179.900000,-0.102645,bg,"1,11",1169543,Раница Rains Trail Cord Rolltop Backpack W3 в ...,41599
4,"230,232",0,Rains Batoh 14480 Černá,77.959184,-0.240293,cz,11,671883,Batoh Rains,41599
...,...,...,...,...,...,...,...,...,...,...
928229,"775,1778",0,Modelka má na sobě velikost jedna. Míry modelk...,19.551020,-0.812110,cz,1,480141,Top DHJ-TP-17733.22X ITALY MODA,65996
928230,0,0,Kupuj Detský ruksak Simpsonovci – hnedá na adi...,28.000000,-0.366371,sk,42,1278444,Adidas Detský ruksak Simpsonovci,214495
928231,6444,0,- Polyester použitý na výrobu tohto modelu je ...,28.990000,-0.351128,sk,42,439823,"Detský ruksak adidas Performance čierna farba,...",214495
928232,6444,0,Dámske šľapky značky Only.\n- Model: Morgan-1\...,33.790000,-0.277221,sk,1,34958,Dámske šľapky Only,140959


In [18]:
pickle.dump(pipeline, open(os.path.join(pipelines_folder, 'preprocessing_pipeline.pkl'), 'wb'))

## V tomto stavu:
- Nejsou null hodnoty ve sloupcích:
    - colorTagIdString
    - description
    -  brandEditionTagId
- Barvy a deptId maji stejny format - cisla oddelena carkou
- chybi udelat OneHot encodingy kategorickejch - to udelame v PyTorchi